# CNN + GloVe (starter) — Jigsaw toxic comments

**Preprocessing:** `preprocess_for_cnn` (word indices, train-only vocab).

**Default in this notebook:** **full baseline** — entire train split (after stratified train/val), `MAX_LEN = 256`, multi-epoch training, no row caps. Other modes remain available: **quick** smoke (`QUICK_ITERATION = True`) or **progress report** (`PROGRESS_REPORT_MODE = True`).

**Next steps for your proposal:** Download GloVe (e.g. `glove.6B.100d.txt`), implement `build_glove_weight_matrix(vocab, path, dim)` aligned to `data.vocab`, pass `embedding.weight.data.copy_(...)`. Until then this notebook uses **trainable** random embeddings so the pipeline runs.

**Metrics (course proposal):** micro/macro F1, precision/recall/ROC-AUC per label, confusion matrices, training time, parameter count. Validation predictions and metrics are saved under `notebooks/cnn_baseline_outputs/`: `cnn_baseline_*` when `USE_POS_WEIGHT=False`, `cnn_posweight_*` when `USE_POS_WEIGHT=True` (train-only `pos_weight` in `BCEWithLogitsLoss`). With `USE_POS_WEIGHT=True`, the eval cell also runs **per-label threshold tuning** on the validation set (max F1 per label, not 0.5), prints a three-way comparison vs baseline `@0.5`, and stores `threshold_tuned_eval` plus `y_pred_tuned` / `best_threshold_per_label` in the artifacts.

In [1]:
from pathlib import Path
import sys

# Repo root (contains preprocessing/)
ROOT = Path.cwd().resolve()
for c in (ROOT, ROOT.parent):
    if (c / "preprocessing" / "text_preprocessing.py").exists():
        ROOT = c
        break
sys.path.insert(0, str(ROOT))
# notebooks/ for metrics_helpers
sys.path.insert(0, str(ROOT / "notebooks"))

In [2]:
import time

import numpy as np
import torch
import torch.nn as nn
from IPython.display import display
from torch.utils.data import DataLoader, TensorDataset

from preprocessing.text_preprocessing import LABEL_COLUMNS
from metrics_helpers import multilabel_evaluation_report, per_label_confusion_matrices, torch_parameter_count

def pick_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


DEVICE = pick_device()
RNG = np.random.default_rng(42)
torch.manual_seed(42)

from preprocessing.text_preprocessing import preprocess_for_cnn

# ========== Run configuration (edit here) ==========
# Three modes (quick wins if True):
#   1) quick smoke     — QUICK_ITERATION = True
#   2) progress report — QUICK_ITERATION = False, PROGRESS_REPORT_MODE = True
#   3) full              — QUICK_ITERATION = False, PROGRESS_REPORT_MODE = False
QUICK_ITERATION = False
PROGRESS_REPORT_MODE = False

# Class imbalance: BCEWithLogitsLoss(pos_weight) from train-only n_neg/n_pos per label. False = baseline loss.
USE_POS_WEIGHT = True

LR = 1e-3

# --- Quick smoke-test (used only if QUICK_ITERATION) ---
QUICK_MAX_TRAIN_SAMPLES = 2048
QUICK_MAX_VAL_SAMPLES = 512
QUICK_MAX_LEN = 64
QUICK_BATCH_SIZE = 128
QUICK_EPOCHS = 1
QUICK_MAX_VOCAB = 3000
QUICK_MIN_FREQ = 1

# --- Progress-report run (used if not quick and PROGRESS_REPORT_MODE) ---
PROGRESS_MAX_TRAIN_SAMPLES = 10_000
PROGRESS_MAX_VAL_SAMPLES = 2_000
PROGRESS_MAX_LEN = 128
PROGRESS_BATCH_SIZE = 64
PROGRESS_EPOCHS = 2
PROGRESS_MAX_VOCAB = 10_000
PROGRESS_MIN_FREQ = 2

# --- Full run (both flags False) ---
FULL_MAX_TRAIN_SAMPLES = None
FULL_MAX_VAL_SAMPLES = None
FULL_MAX_LEN = 256
FULL_BATCH_SIZE = 64
FULL_EPOCHS = 7
FULL_MAX_VOCAB = 50_000
FULL_MIN_FREQ = 2

if QUICK_ITERATION:
    run_mode = "quick (smoke test)"
    _train_n = QUICK_MAX_TRAIN_SAMPLES
    _val_n = QUICK_MAX_VAL_SAMPLES
    MAX_LEN = QUICK_MAX_LEN
    BATCH_SIZE = QUICK_BATCH_SIZE
    EPOCHS = QUICK_EPOCHS
    MAX_VOCAB = QUICK_MAX_VOCAB
    MIN_FREQ = QUICK_MIN_FREQ
elif PROGRESS_REPORT_MODE:
    run_mode = "progress report"
    _train_n = PROGRESS_MAX_TRAIN_SAMPLES
    _val_n = PROGRESS_MAX_VAL_SAMPLES
    MAX_LEN = PROGRESS_MAX_LEN
    BATCH_SIZE = PROGRESS_BATCH_SIZE
    EPOCHS = PROGRESS_EPOCHS
    MAX_VOCAB = PROGRESS_MAX_VOCAB
    MIN_FREQ = PROGRESS_MIN_FREQ
else:
    run_mode = "full"
    _train_n = FULL_MAX_TRAIN_SAMPLES
    _val_n = FULL_MAX_VAL_SAMPLES
    MAX_LEN = FULL_MAX_LEN
    BATCH_SIZE = FULL_BATCH_SIZE
    EPOCHS = FULL_EPOCHS
    MAX_VOCAB = FULL_MAX_VOCAB
    MIN_FREQ = FULL_MIN_FREQ

data = preprocess_for_cnn(
    max_len=MAX_LEN,
    validation_fraction=0.1,
    random_state=42,
    min_freq=MIN_FREQ,
    max_vocab=MAX_VOCAB,
    max_train_samples=_train_n,
    max_val_samples=_val_n,
)
vocab_size = len(data.vocab)
embed_dim = 100
num_labels = len(LABEL_COLUMNS)

# Train-only pos_weight = n_negative / n_positive per label (for optional BCEWithLogitsLoss).
_y_train = np.asarray(data.y_train, dtype=np.float64)
pos_weight_train = np.zeros(num_labels, dtype=np.float64)
for _i in range(num_labels):
    _pos = float(_y_train[:, _i].sum())
    _neg = float(_y_train.shape[0] - _pos)
    pos_weight_train[_i] = (_neg / _pos) if _pos > 0 else 1.0
POS_WEIGHT = (
    torch.tensor(pos_weight_train, dtype=torch.float32, device=DEVICE)
    if USE_POS_WEIGHT
    else None
)

n_train, n_val = data.X_train.shape[0], data.X_val.shape[0]
print("=" * 60)
print("CONFIG")
print("  mode:", run_mode)
print("  device:", DEVICE)
print("  USE_POS_WEIGHT:", USE_POS_WEIGHT)
print("  train_samples:", n_train, "| val_samples:", n_val)
print("  MAX_LEN:", MAX_LEN, "| vocab_size:", vocab_size)
print("  BATCH_SIZE:", BATCH_SIZE, "| EPOCHS:", EPOCHS)
print("=" * 60)

CONFIG
  mode: full
  device: cpu
  USE_POS_WEIGHT: True
  train_samples: 143613 | val_samples: 15958
  MAX_LEN: 256 | vocab_size: 50000
  BATCH_SIZE: 64 | EPOCHS: 7


In [3]:
class TextCNN(nn.Module):
    # Kim-style multi-channel CNN over embeddings

    def __init__(
        self,
        vocab_size: int,
        embed_dim: int,
        num_labels: int,
        padding_idx: int = 0,
        kernel_sizes: tuple[int, ...] = (3, 4, 5),
        num_filters: int = 100,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        self.convs = nn.ModuleList([nn.Conv1d(embed_dim, num_filters, k) for k in kernel_sizes])
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), num_labels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        e = self.embedding(x).transpose(1, 2)
        pools = []
        for conv in self.convs:
            h = torch.relu(conv(e))
            pools.append(h.max(dim=2).values)
        h = torch.cat(pools, dim=1)
        return self.fc(self.dropout(h))


model = TextCNN(vocab_size, embed_dim, num_labels, padding_idx=0).to(DEVICE)
if USE_POS_WEIGHT:
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT)
else:
    loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

print(
    "Training loss: BCEWithLogitsLoss —",
    "pos_weight ON (train-only n_neg/n_pos per label)"
    if USE_POS_WEIGHT
    else "pos_weight OFF (baseline)",
    flush=True,
)
if USE_POS_WEIGHT:
    for _name, _w in zip(LABEL_COLUMNS, pos_weight_train):
        print(f"  {_name}: {_w:.4f}", flush=True)

train_ds = TensorDataset(
    torch.tensor(data.X_train, dtype=torch.long),
    torch.tensor(data.y_train, dtype=torch.float32),
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

val_ds = TensorDataset(
    torch.tensor(data.X_val, dtype=torch.long),
    torch.tensor(data.y_val, dtype=torch.float32),
)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

t0 = time.perf_counter()
model.train()
final_train_loss = 0.0
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    n_batches = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1
    final_train_loss = epoch_loss / max(n_batches, 1)
    print(
        f"Epoch {epoch + 1}/{EPOCHS} | train loss: {final_train_loss:.4f} | elapsed: {time.perf_counter() - t0:.1f}s",
        flush=True,
    )
train_seconds = time.perf_counter() - t0

model.eval()
val_loss_sum = 0.0
val_batches = 0
with torch.no_grad():
    for xb, yb in val_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        val_loss_sum += loss_fn(model(xb), yb).item()
        val_batches += 1
val_loss = val_loss_sum / max(val_batches, 1)

print(f"Training wall time ({EPOCHS} epoch(s)): {train_seconds:.1f} s")
print(f"Final train loss (last epoch avg): {final_train_loss:.4f} | val loss: {val_loss:.4f}")

Training loss: BCEWithLogitsLoss — pos_weight ON (train-only n_neg/n_pos per label)
  toxic: 9.4423
  severe_toxic: 99.2884
  obscene: 17.8493
  threat: 336.9129
  insult: 19.2443
  identity_hate: 112.7078
Epoch 1/7 | train loss: 0.8401 | elapsed: 521.9s
Epoch 2/7 | train loss: 0.5171 | elapsed: 854.5s
Epoch 3/7 | train loss: 0.4138 | elapsed: 1189.9s
Epoch 4/7 | train loss: 0.3437 | elapsed: 1519.3s
Epoch 5/7 | train loss: 0.3061 | elapsed: 1843.0s
Epoch 6/7 | train loss: 0.2877 | elapsed: 2168.7s
Epoch 7/7 | train loss: 0.2592 | elapsed: 2491.2s
Training wall time (7 epoch(s)): 2491.2 s
Final train loss (last epoch avg): 0.2592 | val loss: 0.6945


In [4]:
import json

from sklearn.metrics import f1_score as sk_f1


def predict_logits(model, X_np: np.ndarray, batch_size: int = 512) -> np.ndarray:
    model.eval()
    out = []
    x = torch.tensor(X_np, dtype=torch.long)
    with torch.no_grad():
        for i in range(0, len(x), batch_size):
            logits = model(x[i : i + batch_size].to(DEVICE))
            out.append(logits.cpu().numpy())
    return np.concatenate(out, axis=0)


# Raw validation inference time (model forward + sigmoid only; excludes threshold search)
infer_t0 = time.perf_counter()
logits_val = predict_logits(model, data.X_val)
prob_val = torch.sigmoid(torch.from_numpy(logits_val)).numpy()
inference_time_s = time.perf_counter() - infer_t0
pred_val_05 = (prob_val >= 0.5).astype(np.int64)
pred_val = pred_val_05

y_val_arr = np.asarray(data.y_val, dtype=np.float32)

per_label_df_05, summary_05 = multilabel_evaluation_report(
    y_val_arr, pred_val_05, prob_val, LABEL_COLUMNS
)
per_label_df, summary = per_label_df_05, summary_05
display(per_label_df_05)
confs = per_label_confusion_matrices(y_val_arr, pred_val_05, LABEL_COLUMNS)
for name, m in confs.items():
    print(name, "\n", m)

print()
_val_title = "CNN + pos_weight" if USE_POS_WEIGHT else "CNN baseline"
print(f"=== Validation @ 0.5 threshold — {_val_title} ===")
print(f"  Total training time: {train_seconds:.2f} s ({EPOCHS} epoch(s))")
print(f"  Micro F1:  {summary_05['f1_micro']:.4f}")
print(f"  Macro F1: {summary_05['f1_macro']:.4f}")
print("  Per-label F1:")
for _, row in per_label_df_05.iterrows():
    print(f"    {row['label']}: {row['f1']:.4f}")

print()
print("--- Proposal summary (record for report / comparison) ---")
print(f"  device: {DEVICE}")
print(f"  training_time_s: {train_seconds:.2f}")
print(f"  inference_time_s: {inference_time_s:.2f}")
print(f"  num_parameters: {torch_parameter_count(model)}")
if summary_05["f1_macro"] == 0.0 and summary_05["f1_micro"] == 0.0:
    print(
        "  Note: F1 can be 0 if every predicted label is 0 at threshold 0.5 (common with very few epochs)."
        " ROC-AUC may still be > 0.5. Use more EPOCHS or check data / logits scale."
    )

print()
print("--- Run summary ---")
print(f"  mode: {run_mode}")
print(f"  final_train_loss: {final_train_loss:.4f}")
print(f"  val_loss: {val_loss:.4f}")
print(f"  inference_time_s: {inference_time_s:.2f}")

tuned_threshold_eval = None
pred_val_tuned = None
best_thresholds_list = None
threshold_tuning_time_s = None

if USE_POS_WEIGHT:
    tune_t0 = time.perf_counter()
    y_int = y_val_arr.astype(np.int64)
    ths = np.linspace(0.01, 0.99, 99)
    best_meta = []
    pred_tuned = np.zeros_like(y_int, dtype=np.int64)
    for i, lbl in enumerate(LABEL_COLUMNS):
        yt = y_int[:, i]
        yp_col = prob_val[:, i]
        best_f, best_th = -1.0, 0.5
        for t in ths:
            pr = (yp_col >= t).astype(np.int64)
            f = sk_f1(yt, pr, zero_division=0)
            if f > best_f + 1e-12 or (
                abs(f - best_f) <= 1e-12 and abs(float(t) - 0.5) < abs(best_th - 0.5)
            ):
                best_f, best_th = float(f), float(t)
        best_meta.append({"label": lbl, "threshold": best_th})
        pred_tuned[:, i] = (yp_col >= best_th).astype(np.int64)

    per_label_df_tuned, summary_tuned = multilabel_evaluation_report(
        y_val_arr, pred_tuned, prob_val, LABEL_COLUMNS
    )
    pred_val_tuned = pred_tuned

    print()
    print("=== Per-label threshold tuning (val, maximize F1 per label; not 0.5) ===")
    for bm, (_, row) in zip(best_meta, per_label_df_tuned.iterrows()):
        print(
            f"  {bm['label']}: threshold={bm['threshold']:.4f} | "
            f"F1={float(row['f1']):.4f} P={float(row['precision']):.4f} R={float(row['recall']):.4f}"
        )
    print(f"  Micro F1 (tuned):  {summary_tuned['f1_micro']:.4f}")
    print(f"  Macro F1 (tuned): {summary_tuned['f1_macro']:.4f}")

    _baseline_json = ROOT / "notebooks" / "cnn_baseline_outputs" / "cnn_baseline_metrics.json"
    if _baseline_json.is_file():
        _bm = json.loads(_baseline_json.read_text(encoding="utf-8"))
        print()
        print("--- Comparison (validation) ---")
        print(
            f"  Baseline CNN @0.5:        micro {_bm['f1_micro']:.4f} | macro {_bm['f1_macro']:.4f}"
        )
        print(
            f"  pos_weight CNN @0.5:      micro {summary_05['f1_micro']:.4f} | macro {summary_05['f1_macro']:.4f}"
        )
        print(
            f"  pos_weight + tuned thr:   micro {summary_tuned['f1_micro']:.4f} | macro {summary_tuned['f1_macro']:.4f}"
        )

    threshold_tuning_time_s = time.perf_counter() - tune_t0
    print(f"  Threshold tuning time: {threshold_tuning_time_s:.2f} s")

    tuned_threshold_eval = {
        "best_threshold_per_label": {b["label"]: float(b["threshold"]) for b in best_meta},
        "f1_micro": float(summary_tuned["f1_micro"]),
        "f1_macro": float(summary_tuned["f1_macro"]),
        "threshold_tuning_time_s": float(threshold_tuning_time_s),
        "per_label": [
            {
                "label": str(row["label"]),
                "precision": float(row["precision"]),
                "recall": float(row["recall"]),
                "f1": float(row["f1"]),
                "roc_auc": float(row["roc_auc"]),
            }
            for _, row in per_label_df_tuned.iterrows()
        ],
    }
    best_thresholds_list = [float(b["threshold"]) for b in best_meta]

# Save validation predictions + metrics for later comparison
out_dir = ROOT / "notebooks" / "cnn_baseline_outputs"
out_dir.mkdir(parents=True, exist_ok=True)
_artifact_stem = "cnn_posweight" if USE_POS_WEIGHT else "cnn_baseline"
val_npz = out_dir / f"{_artifact_stem}_val.npz"
_save_npz = dict(
    y_true=y_val_arr,
    y_pred=pred_val_05.astype(np.int64),
    y_prob=prob_val.astype(np.float32),
    label_names=np.array(LABEL_COLUMNS, dtype=object),
)
if USE_POS_WEIGHT and pred_val_tuned is not None:
    _save_npz["y_pred_tuned"] = pred_val_tuned.astype(np.int64)
    _save_npz["best_threshold_per_label"] = np.array(
        best_thresholds_list, dtype=np.float32
    )
np.savez_compressed(val_npz, **_save_npz)
metrics_export = {
    "model": "TextCNN",
    "loss": "BCEWithLogitsLoss",
    "use_pos_weight": bool(USE_POS_WEIGHT),
    "pos_weight_per_label": {
        str(LABEL_COLUMNS[i]): float(pos_weight_train[i]) for i in range(len(LABEL_COLUMNS))
    },
    "run_mode": run_mode,
    "device": str(DEVICE),
    "training_time_s": float(train_seconds),
    "inference_time_s": float(inference_time_s),
    "epochs": int(EPOCHS),
    "config": {
        "max_len": int(MAX_LEN),
        "batch_size": int(BATCH_SIZE),
        "lr": float(LR),
        "min_freq": int(MIN_FREQ),
        "max_vocab": int(MAX_VOCAB),
        "validation_fraction": 0.1,
        "random_state": 42,
    },
    "train_samples": int(n_train),
    "val_samples": int(n_val),
    "final_train_loss": float(final_train_loss),
    "val_loss": float(val_loss),
    "f1_micro": float(summary_05["f1_micro"]),
    "f1_macro": float(summary_05["f1_macro"]),
    "per_label_f1": {
        str(row["label"]): float(row["f1"]) for _, row in per_label_df_05.iterrows()
    },
    "per_label": [
        {
            "label": str(row["label"]),
            "precision": float(row["precision"]),
            "recall": float(row["recall"]),
            "f1": float(row["f1"]),
            "roc_auc": float(row["roc_auc"]),
        }
        for _, row in per_label_df_05.iterrows()
    ],
    "num_parameters": int(torch_parameter_count(model)),
}
if USE_POS_WEIGHT and tuned_threshold_eval is not None:
    metrics_export["threshold_tuning_time_s"] = float(threshold_tuning_time_s)
    metrics_export["threshold_tuned_eval"] = tuned_threshold_eval
metrics_json = out_dir / f"{_artifact_stem}_metrics.json"
with open(metrics_json, "w", encoding="utf-8") as f:
    json.dump(metrics_export, f, indent=2)
print()
print("Saved:", val_npz)
print("Saved:", metrics_json)

,label,precision,recall,f1,roc_auc
0,toxic,0.532703,0.840363,0.652064,0.950094
1,severe_toxic,0.222756,0.852761,0.353240,0.976085
2,obscene,0.528686,0.877108,0.659719,0.976232
3,threat,0.112045,0.754717,0.195122,0.970424
4,insult,0.414166,0.881226,0.563495,0.967278
5,identity_hate,0.116740,0.746479,0.201905,0.949923


toxic 
 [[13281  1136]
 [  246  1295]]
severe_toxic 
 [[15310   485]
 [   24   139]]
obscene 
 [[14479   649]
 [  102   728]]
threat 
 [[15588   317]
 [   13    40]]
insult 
 [[14199   976]
 [   93   690]]
identity_hate 
 [[15014   802]
 [   36   106]]

=== Validation @ 0.5 threshold — CNN + pos_weight ===
  Total training time: 2491.25 s (7 epoch(s))
  Micro F1:  0.5514
  Macro F1: 0.4376
  Per-label F1:
    toxic: 0.6521
    severe_toxic: 0.3532
    obscene: 0.6597
    threat: 0.1951
    insult: 0.5635
    identity_hate: 0.2019

--- Proposal summary (record for report / comparison) ---
  device: cpu
  training_time_s: 2491.25
  num_parameters: 5122106

--- Run summary ---
  mode: full
  final_train_loss: 0.2592
  val_loss: 0.6945

=== Per-label threshold tuning (val, maximize F1 per label; not 0.5) ===
  toxic: threshold=0.8800 | F1=0.7265 P=0.7625 R=0.6937
  severe_toxic: threshold=0.9700 | F1=0.4794 P=0.3960 R=0.6074
  obscene: threshold=0.9400 | F1=0.7526 P=0.7595 R=0.7458
  threa

## Proposal checklist (report)

- Compare **training time** and **parameter count** across the four model notebooks.
- **Per-label threshold tuning** on validation (default here: 0.5) for rare classes such as `threat`.
- **Model size on disk**: save `state_dict` or full checkpoint and report file size in MB.
- Optional: **macro** vs **micro** trade-offs given imbalance (already reported above).